# 🚁 SGLATrack: CVPR 2025 Vision Transformer Pipeline (AIC-4)
This notebook implements the **SGLATrack** tracker with full compatibility for PyTorch 2.6+ and the complete UAV123/20L dataset.

### 📌 Features:
- **CVPR 2025 Architecture**: Advanced Similarity-Guided Transformer.
- **PyTorch 2.0+ Compatibility**: Patched `torch._six` and `UnpicklingError`.
- **Weights**: Uses the specific `sglatrack_ep0297.pth.tar` checkpoint.
- **Full Dataset Discovery**: Hits the 117k+ frame target.

In [ ]:
# --- 0. Install Dependencies ---
!pip install yacs tensorboardX easydict shapely numba timm==0.5.4 --quiet
print("✅ Dependencies installed.")

In [6]:
import sys
import collections.abc
import functools
import argparse
from types import ModuleType
import os
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset
from tqdm.notebook import tqdm
import glob
from easydict import EasyDict as edict
from collections import OrderedDict

# --- 1. Compatibility Patches ---
if 'torch._six' not in sys.modules:
    _six = ModuleType("torch._six")
    _six.container_abcs = collections.abc
    sys.modules["torch._six"] = _six
    print("✅ Patched torch._six")

try:
    from torch.serialization import add_safe_globals
    add_safe_globals([argparse.Namespace, np.core.multiarray.scalar, np._core.multiarray.scalar])
    print("✅ Added safe globals")
except:
    pass

if not hasattr(torch, '__patched_for_aic'):
    _orig_load = torch.load
    @functools.wraps(_orig_load)
    def _patched_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = _patched_load
    torch.__patched_for_aic = True
    print("✅ torch.load patched")

# --- 2. Robust Path Discovery ---
def discover_paths():
    # ✅ Fix 1: these are plain strings, not lists — remove [0] indexing
    code    = '/kaggle/input/datasets/michealehab4/sglatrack-code/SGLATrack-main'
    weights = '/kaggle/input/datasets/michealehab4/sglatrack-weights/sglatrack_ep0297.pth.tar'
    data    = '/kaggle/input/datasets/beshoyzaki/aic4-uav123-full/dataset/UAV123'

    res = {}

    if os.path.exists(code):
        res['code'] = code          # ✅ Fix 2: no [0], it's already a string
        sys.path.insert(0, res['code'])
        sys.path.insert(0, os.path.join(res['code'], 'lib'))
        print(f"✅ Code: {res['code']}")
    else:
        print(f"❌ Code path not found: {code}")

    if os.path.exists(weights):
        res['weights'] = weights    # ✅ Fix 2: no [0]
        print(f"✅ Weights: {res['weights']}")
    else:
        print(f"❌ Weights not found: {weights}")

    if os.path.exists(data):
        res['data'] = data          # ✅ Fix 2: no [0]
        print(f"✅ Data: {res['data']}")
    else:
        print(f"❌ Data path not found: {data}")

    return res

PATHS = discover_paths()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- 3. Setup Local Config ---
# ✅ Fix 3: write to /kaggle/working (writable), NOT to /kaggle/input (read-only)
# ✅ Fix 4: removed bad indentation — this is now top-level code, not inside a function

code = PATHS.get('code', '')
local_py_dir = os.path.join('/kaggle/working', 'lib', 'test', 'evaluation')
os.makedirs(local_py_dir, exist_ok=True)

local_py_path = os.path.join(local_py_dir, 'local.py')
with open(local_py_path, 'w') as f:
    f.write(f"""from lib.test.evaluation.environment import EnvSettings

def local_env_settings():
    settings = EnvSettings()
    settings.prj_dir = '{code}'
    settings.save_dir = '/kaggle/working'
    return settings
""")

# Also make sure /kaggle/working is in sys.path so the written local.py can be found
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

print(f"✅ local.py written to: {local_py_path}")

✅ Patched torch._six
✅ Added safe globals
✅ torch.load patched
✅ Code: /kaggle/input/datasets/michealehab4/sglatrack-code/SGLATrack-main
✅ Weights: /kaggle/input/datasets/michealehab4/sglatrack-weights/sglatrack_ep0297.pth.tar
✅ Data: /kaggle/input/datasets/beshoyzaki/aic4-uav123-full/dataset/UAV123
Device: cpu
✅ local.py written to: /kaggle/working/lib/test/evaluation/local.py


In [2]:
# ============================================================
# FULL SETUP — Run this single cell after any kernel restart
# ============================================================
import sys, os, json, glob, cv2, math
import collections.abc, functools, argparse, importlib
import numpy as np
from types import ModuleType, SimpleNamespace
from tqdm.notebook import tqdm
from easydict import EasyDict as edict
import torch

# --- Paths ---
CODE      = '/kaggle/input/datasets/michealehab4/sglatrack-code/SGLATrack-main'
WEIGHTS   = '/kaggle/input/datasets/michealehab4/sglatrack-weights/sglatrack_ep0297.pth.tar'
DATA_ROOT = '/kaggle/input/datasets/georgewanis/aic-competition-data'
MANIFEST  = f"{DATA_ROOT}/metadata/contestant_manifest.json"

# --- sys.path ---
sys.path = [p for p in sys.path if 'SGLATrack' not in p]
sys.path.insert(0, CODE)
print(f"✅ sys.path[0] = {CODE}")

# --- torch._six patch ---
_six = ModuleType("torch._six")
_six.container_abcs = collections.abc
_six.string_classes  = (str,)
_six.int_classes     = (int,)
_six.inf             = float('inf')
sys.modules["torch._six"] = _six

# --- torch.load patch ---
if not hasattr(torch, '__patched_for_aic'):
    _orig_load = torch.load
    @functools.wraps(_orig_load)
    def _patched_load(*args, **kwargs):
        if 'weights_only' not in kwargs: kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = _patched_load
    torch.__patched_for_aic = True

# --- safe globals ---
try:
    from torch.serialization import add_safe_globals
    add_safe_globals([argparse.Namespace,
                      np.core.multiarray.scalar,
                      np._core.multiarray.scalar])
except: pass

# --- Mock optional deps ---
for name, attrs in [
    ("visdom",             {"Visdom": type("Visdom", (), {"__init__": lambda s,*a,**k: None})}),
    ("visdom.server",      {}),
    ("jpeg4py",            {"JPEG": type("JPEG", (), {"__init__": lambda s,*a,**k: None, "decode": lambda s: None})}),
    ("tikzplotlib",        {}),
    ("wandb",              {}),
    ("lmdb",               {}),
    ("pycocotools",        {}),
    ("pycocotools.coco",   {}),
]:
    mod = ModuleType(name)
    for k, v in attrs.items(): setattr(mod, k, v)
    sys.modules.setdefault(name, mod)

# --- Purge stale lib cache ---
stale = [k for k in sys.modules if k == 'lib' or k.startswith('lib.')]
for k in stale: del sys.modules[k]
print(f"🧹 Purged {len(stale)} stale lib modules")

# --- Monkey-patch env_settings ---
import lib.test.evaluation.environment as env_mod
def patched_env_settings():
    s = SimpleNamespace()
    s.prj_dir  = CODE
    s.save_dir = '/kaggle/working'
    return s
env_mod.env_settings = patched_env_settings
print("✅ env_settings patched")

# --- Import SGLATrack ---
from lib.test.tracker.sglatrack import sglatrack as SGLATracker
from lib.test.parameter.sglatrack import parameters
print("✅ SGLATrack imported")

# --- Load model ---
params = parameters('deit_distilled')
params.checkpoint     = WEIGHTS
params.debug          = False
params.save_all_boxes = False

tracker = SGLATracker(params, 'uav123')
total   = sum(p.numel() for p in tracker.network.parameters()) / 1e6
device  = next(tracker.network.parameters()).device
print(f"✅ SGLATrack loaded — {total:.2f}M params on {device}")

# ============================================================
# Dataset
# ============================================================
class AICDataset:
    def __init__(self, root, manifest_path, split='public_lb'):  # ✅ default to public_lb
        with open(manifest_path, 'r') as f:
            manifest = json.load(f)

        # ✅ Only load the requested split
        if split not in manifest:
            raise ValueError(f"Split '{split}' not found. Available: {list(manifest.keys())}")

        self.sequences = []
        for seq_key, info in manifest[split].items():
            video_path = os.path.join(root, info['video_path'])
            anno_path  = os.path.join(root, info['annotation_path'])

            if not os.path.exists(video_path):
                print(f"⚠️  Missing video: {video_path}")
                continue
            if not os.path.exists(anno_path):
                print(f"⚠️  Missing anno:  {anno_path}")
                continue

            with open(anno_path, 'r') as af:
                line = af.readline().strip()
            parts = [float(x) for x in line.replace('\t',' ').replace(',',' ').split() if x]

            self.sequences.append({
                'key':      seq_key,           # e.g. "dataset1/Car_video"
                'video':    video_path,
                'n_frames': info['n_frames'],
                'gt_bbox':  parts[:4],         # [x, y, w, h] first frame only
            })

        self.sequences.sort(key=lambda x: x['key'])
        total_frames = sum(s['n_frames'] for s in self.sequences)
        print(f"🚀 Split='{split}' → {len(self.sequences)} sequences, {total_frames:,} frames")

    def __len__(self): return len(self.sequences)
# ============================================================
# Submission
# ============================================================
def generate_submission(tracker, dataset):
    results = ["id,x,y,w,h\n"]
    total_rows = 0
    skipped    = 0

    for seq in tqdm(dataset.sequences, desc="Tracking"):
        key, gt, n_frames = seq['key'], seq['gt_bbox'], seq['n_frames']
        cap = cv2.VideoCapture(seq['video'])
        if not cap.isOpened():
            print(f"❌ Cannot open: {seq['video']}")
            skipped += 1
            continue

        initialized = False
        pred = gt

        for idx in range(n_frames):
            ret, frame_bgr = cap.read()
            if not ret:
                results.append(f"{key}_{idx},{pred[0]:.3f},{pred[1]:.3f},{pred[2]:.3f},{pred[3]:.3f}\n")
                total_rows += 1
                continue

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            H, W = frame_rgb.shape[:2]

            if idx == 0:
                x = max(0.0, min(gt[0], W - 1))
                y = max(0.0, min(gt[1], H - 1))
                w = max(10.0, min(gt[2], W - x))
                h = max(10.0, min(gt[3], H - y))
                x = min(x, W - w)
                y = min(y, H - h)
                try:
                    tracker.initialize(frame_rgb, {'init_bbox': [x, y, w, h]})
                    pred = [x, y, w, h]
                    initialized = True
                except Exception as e:
                    print(f"⚠️  [{key}] init failed: {e}")
                    pred = [x, y, w, h]
                    skipped += 1
            else:
                if initialized:
                    try:
                        pred = tracker.track(frame_rgb)['target_bbox']
                    except Exception as e:
                        print(f"⚠️  [{key}] frame {idx}: {e}")

            results.append(f"{key}_{idx},{pred[0]:.3f},{pred[1]:.3f},{pred[2]:.3f},{pred[3]:.3f}\n")
            total_rows += 1

        cap.release()

    with open('/kaggle/working/submission.csv', 'w') as f:
        f.writelines(results)
    print(f"\n✅ Done — {total_rows:,} rows | {skipped} skipped")


# --- Run ---
dataset = AICDataset(DATA_ROOT, MANIFEST)
generate_submission(tracker, dataset)

✅ sys.path[0] = /kaggle/input/datasets/michealehab4/sglatrack-code/SGLATrack-main
🧹 Purged 53 stale lib modules
✅ env_settings patched
✅ SGLATrack imported
test config:  {'MODEL': {'PRETRAIN_FILE': 'deit_tiny_distilled_patch16_224.pth', 'EXTRA_MERGER': False, 'RETURN_INTER': False, 'RETURN_STAGES': [], 'BACKBONE': {'TYPE': 'deit_tiny_distilled_patch16', 'STRIDE': 16, 'MID_PE': False, 'SEP_SEG': False, 'CAT_MODE': 'direct', 'MERGE_LAYER': 0, 'ADD_CLS_TOKEN': False, 'CLS_TOKEN_USE_MODE': 'ignore', 'CE_LOC': [], 'CE_KEEP_RATIO': [], 'CE_TEMPLATE_RANGE': 'ALL'}, 'HEAD': {'TYPE': 'CENTER', 'NUM_CHANNELS': 256}}, 'TRAIN': {'LR': 0.0004, 'WEIGHT_DECAY': 0.0001, 'EPOCH': 300, 'LR_DROP_EPOCH': 240, 'BATCH_SIZE': 32, 'NUM_WORKER': 4, 'OPTIMIZER': 'ADAMW', 'BACKBONE_MULTIPLIER': 0.1, 'GIOU_WEIGHT': 2.0, 'L1_WEIGHT': 5.0, 'FREEZE_LAYERS': [0], 'PRINT_INTERVAL': 50, 'VAL_EPOCH_INTERVAL': 999, 'GRAD_CLIP_NORM': 0.1, 'AMP': False, 'CE_START_EPOCH': 20, 'CE_WARM_EPOCH': 80, 'DROP_PATH_RATE': 0.1, 'SCH

Tracking:   0%|          | 0/89 [00:00<?, ?it/s]


✅ Done — 74,293 rows | 0 skipped
